## Text Extraction

In [5]:
import pandas as pd

In [6]:
from google.colab import files

file=files.upload()

Saving ICMR_Dataset.pdf to ICMR_Dataset.pdf


In [7]:
! pip install PyMuPDF

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 60.3 MB/s eta 0:00:00


In [8]:
import fitz # for pdf read

In [9]:
pdf_doc=fitz.open('/content/ICMR_Dataset.pdf')
len(pdf_doc)

54

In [10]:
data=[]

for idx in range(23,50):

  page=pdf_doc.load_page(idx)
  text=page.get_text()

  data.append({'page':idx-22, 'text':text})

pdf_doc.close()

In [11]:
df=pd.DataFrame(data)
df.head()

,page,text
0,1,19\n1.\t Terminal illness \nAccording to the I...
1,2,20\nantibiotics when given for a potentially \...
2,3,21\nsigned and dated to be valid. In many \nse...
3,4,22\ncosts then the opportunity of a trial of \...
4,5,23\nterm passive euthanasia but using \nthe mo...


In [12]:
df.to_csv('ICMR_Dataset.csv', index=False)

## Text Cleaning

In [13]:
import re

In [14]:
for id in range(len(df)):
  txt=df.loc[id,'text']

  txt=re.sub(r'\n+','\n',txt)
  txt=re.sub(r'\t+',' ',txt)
  txt=re.sub(r'\s+',' ',txt)
  txt=txt.strip()

  df.loc[id,'text']=txt

In [15]:
df.to_csv('ICMR_Dataset_cleaned.csv', index=False)

## Chunking

In [16]:
list_df=df['text'].tolist()
text_full=" ".join(list_df)
text_full[:500]

'19 1. Terminal illness According to the International Association of Hospice and Palliative Care, terminal illness is a progressive condition that has no cure and that can be reasonably expected to cause the death of a person within a foreseeable future. The definition is inclusive of both malignant and non- malignant conditions and aging. A person has an eventually fatal condition, if his/ her death in the foreseeable future would not be a surprise. The terms eventually fatal or terminal condit'

In [17]:
words=text_full.split()
len(words), words[0:5]

(8910, ['19', '1.', 'Terminal', 'illness', 'According'])

In [18]:
# so let chunk sizes be 200 and overlap be 50

In [19]:
chunks=[]

start=0
overlap=50
chunk_size=200
end=0

while end<(len(words)):
  end=start+chunk_size

  if(end>len(words)):
    chunks.append(" ".join(words[start:len(words)]))
  else:
    chunks.append(" ".join(words[start:end]))

  start+=(chunk_size-overlap)

In [20]:
len(chunks), chunks[-1]

(60,
 'Lawyers Collective, New Delhi Vasantha Muthuswamy, Formerly at Indian Council of Medical Research, New Delhi Vijay Kumar, Indian Council of Medical Research, New Delhi Vimal Bhandari, National Organ and Tissue Transplant Organization, New Delhi B. ICMR SECRETARIAT Kalyani Thakur, National Centre for Disease Informatics and Research, Bengaluru Rajib Kishore Hazam, National Centre for Disease Informatics and Research, Bengaluru ANNEX 1')

## Word Embeddings

In [21]:
# model: https://huggingface.co/BAAI/bge-base-en-v1.5

In [22]:
!pip install sentence-transformers

In [23]:
from sentence_transformers import SentenceTransformer

In [24]:
model=SentenceTransformer('BAAI/bge-base-en-v1.5')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [25]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

In [26]:
model.to(device)
model

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 768, 'pooling_mode': 'cls', 'include_prompt': True})
  (2): Normalize({})
)

In [27]:
w_embed=model.encode(chunks, convert_to_numpy=True)

In [28]:
w_embed.shape # 60 chunks, each with 768 dim

(60, 768)

In [29]:
w_embed[0].shape, w_embed[0]

((768,),
 array([-3.73755246e-02,  1.81929134e-02,  1.84693895e-02, -2.41598766e-02,
         8.30129385e-02, -2.14033686e-02,  5.09935655e-02,  1.05010802e-02,
        -5.22959977e-03, -5.83333112e-02, -1.27710812e-02, -4.86696586e-02,
        -3.06676328e-02,  4.06681783e-02,  3.16410549e-02,  5.23902811e-02,
         6.84912279e-02, -1.35557831e-03, -2.10793074e-02,  2.53062323e-02,
        -5.23302294e-02, -3.11669651e-02,  7.64276041e-03,  1.35133509e-02,
         5.33387028e-02, -3.00694369e-02, -1.21880872e-02,  3.03406250e-02,
        -5.50445877e-02, -2.52567269e-02, -1.24426819e-02, -2.15868987e-02,
        -2.94898525e-02, -3.10927425e-02,  8.36195517e-03, -1.64115820e-02,
        -3.21390410e-03, -2.13306420e-03, -7.46973185e-03, -5.33555783e-02,
        -8.03352818e-02, -1.12086041e-02,  5.86669100e-03, -4.58565243e-02,
        -2.57356390e-02, -8.66370567e-04,  3.26601490e-02, -1.66082252e-02,
         2.55321693e-02, -3.06689087e-02, -1.02760708e-02, -1.43356959e-03,
   

In [30]:
import numpy as np

cosine=np.dot(w_embed[0],w_embed[1])/(np.linalg.norm(w_embed[0])*np.linalg.norm(w_embed[1]))
cosine

np.float32(0.8352649)

## FAISS for Similarity Search

In [31]:
# reference: https://www.youtube.com/watch?v=0jOlZpFFxCE

In [32]:
! pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 68.6 MB/s eta 0:00:00


In [33]:
import faiss

In [34]:
import numpy as np

In [35]:
dimension=768
index=faiss.IndexFlatL2(dimension)

index.add(w_embed.astype('float32'))

In [36]:
query="What is CPR?"

query_embed=model.encode([query],convert_to_numpy=True)

In [37]:
k=2
distance,indices=index.search(query_embed.astype('float32'),k)

In [38]:
indices

array([[4, 3]])

In [39]:
for indx in indices[0]:
  print(chunks[indx])

5. Cardiopulmonary resuscitation According to the International Liaison Committee on Resuscitation, Cardiopulmonary resuscitation, commonly known as CPR, is an emergency procedure that combines chest compression often with artificial ventilation in an effort to manually preserve intact brain function until further measures are taken to restore spontaneous blood circulation and breathing in a person who is in cardiac arrest. It is indicated in those who are unresponsive with no breathing or abnormal breathing, for example, agonal respirations (30). 6. Do not attempt resuscitation According to the American Heart Association guidelines, a Do Not Attempt Resuscitation (DNAR) order is given by a licensed physician or alternative authority as per local regulation, and it must be 21 signed and dated to be valid. In many settings, “Allow Natural Death” (AND) is becoming a preferred term to replace DNAR, to emphasize that the order is to allow natural consequences of a disease or injury, and to

## Context Generation for Flashcards

In [40]:
query='Palliative Sedation'

In [41]:
query_embed=model.encode([query],convert_to_numpy=True)

In [42]:
k=8

dist,indices=index.search(query_embed.astype('float32'),k)
indices

array([[16, 18, 17, 15, 50, 40, 14, 54]])

In [43]:
context=""

for indx in indices[0]:
  context+=chunks[indx]

In [44]:
context

'to the patient’s level of distress. As with all treatments, patients, when able, should participate in the decision to use palliative sedation. Treatment of other symptoms should be continued alongside palliative sedation, because sedation may decrease the patient’s ability to communicate or display discomfort (37). Palliative sedation raises ethical concerns when it significantly reduces patient consciousness to the degree that the patient is unable to substantially interact with others, does not have the ability or opportunity to change his mind, and is unable to eat and drink (thus potentially shortening survival in particular circumstances). Palliative sedation is ethically defensible when used 1) after careful interdisciplinary evaluation and treatment of the patient, and 2) when palliative treatments that are not intended to affect consciousness have failed or, in the judgment of the clinician, are very likely to fail, 3) where its use is not expected to shorten the patient’s ti

## Flashcard Generation using Qwen

In [45]:
from transformers import pipeline

In [50]:
prompt=f"""
Use ONLY the context mentioned below to generate 5 flashcards.

Context={context}

Flashcards should be structured as:

Question:
Answer:
"""

In [47]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

In [48]:
model_used='Qwen/Qwen2.5-3B-Instruct'

In [49]:
tokenizer=AutoTokenizer.from_pretrained(model_used)
model=AutoModelForCausalLM.from_pretrained(model_used, torch_dtype='auto', device_map='auto')

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [51]:
messages=[
    {
        'role':'system',
        'content':'You are a Medical Professor teaching about terms used in limitation of treatment and providing pallitive end of life care.'
    },
    {
        'role':'user',
        'content':prompt
    }
]

In [54]:
txt=tokenizer.apply_chat_template(messages,tokenize=False, add_generation_prompt=True)

In [55]:
input=tokenizer(txt,return_tensors='pt') # pytorch tensors
input=input.to(device)

In [56]:
output=model.generate(
    input_ids=input.input_ids,
    attention_mask=input.attention_mask,
    max_new_tokens=500,
    do_sample=False,
    temperature=None,
)

In [57]:
output

tensor([[151644,   8948,    198,  ...,   4545,     13, 151645]],
       device='cuda:0')

In [59]:
response=tokenizer.decode(
    output[0][input.input_ids.shape[1]:],
    skip_special_tokens=True
)

In [60]:
print(response)

Card 1:
Question: What is the primary consideration when deciding to use palliative sedation?
Answer: The level of sedation should be proportional to the patient's level of distress.

Card 2:
Question: Under what circumstances might sedation be decreased after a predetermined time?
Answer: Sedation may be decreased after a predetermined time to assess efficacy, continued symptoms, and the need for ongoing sedation.

Card 3:
Question: What are the ethical concerns raised by palliative sedation?
Answer: Ethical concerns include the potential to significantly reduce patient consciousness to the degree that the patient is unable to substantially interact with others, does not have the ability or opportunity to change his mind, and is unable to eat and drink, potentially shortening survival in certain circumstances.

Card 4:
Question: When is palliative sedation ethically defensible?
Answer: Palliative sedation is ethically defensible when used after careful interdisciplinary evaluation and